# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided workflow for loading, exploring, and analyzing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is accessible via a Croissant schema URL and uses standardized identifiers (`@id`) for all entities.

In [ ]:
# Install mlcroissant if not available
!pip install mlcroissant

## 1. Data Loading
We load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata object and display summary
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Let's review the record sets, their fields, and unique `@id`s as defined in the Croissant metadata. These will be referenced for all further operations.

In [ ]:
# List all record sets with their `@id` and field structure
record_sets_info = []
if hasattr(dataset.metadata, 'record_sets'):
    for record_set in dataset.metadata.record_sets:
        print(f"RecordSet Name: {getattr(record_set, 'name', None)} | @id: {record_set.id}")
        if hasattr(record_set, 'fields'):
            print("  Fields:")
            for field in record_set.fields:
                print(f"    - {getattr(field, 'name', None)} (@id: {field.id}, datatype: {getattr(field, 'data_type', None)})")
else:
    print("No record sets defined in metadata.")

## 3. Data Extraction
We can now load the data from each available record set to a Pandas DataFrame, referencing each by their `@id`. You may adjust to select the record set(s) necessary for your analysis.

In [ ]:
# Dynamically gather all record set IDs
record_set_ids = []
if hasattr(dataset.metadata, 'record_sets'):
    record_set_ids = [r.id for r in dataset.metadata.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(df.head())
    print("-"*60)

# If there is only one record set, select it as example
example_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
We apply analysis to a numeric field of interest. All entity references (record set, field) are made by their unique `@id` values.

Typical steps:
- Filtering for values above a threshold in a numeric field
- Normalization (z-score)
- Grouping by a categorical field (if present)

**Please update `<numeric_field_id>` and `<group_field_id>` below to match fields from your chosen record set.**

In [ ]:
# Example IDs (edit as needed with correct @ids for your dataset)
# Use output from the previous cell to find appropriate field IDs.

record_set_id = example_record_set_id

if record_set_id is not None and not dataframes[record_set_id].empty:
    # For illustration: pick first numeric column (float or int)
    df = dataframes[record_set_id]
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
    if len(numeric_cols) > 0:
        numeric_field_id = numeric_cols[0]
        threshold = df[numeric_field_id].mean()  # Or set your own threshold

        # Filter records
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize field (z-score)
        filtered_df[numeric_field_id + "_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

        # Try to group by first non-numeric field, if available
        group_field_id = None
        non_numeric_cols = [col for col in df.columns if col not in numeric_cols]
        if len(non_numeric_cols) > 0:
            group_field_id = non_numeric_cols[0]
        
        if group_field_id is not None:
            print(f"Grouping by {group_field_id} and averaging numeric fields:")
            grouped = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(grouped.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields.

We'll create a histogram and scatter plot for the numeric field (if present). Adjust field IDs as appropriate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the data if available
if record_set_id is not None and not dataframes[record_set_id].empty and len(numeric_cols) > 0:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If another numeric field is available, plot scatter
    if len(numeric_cols) > 1:
        plt.figure(figsize=(7,5))
        sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.title(f"Scatter plot of {numeric_cols[0]} vs {numeric_cols[1]}")
        plt.show()

## 6. Conclusion

In this notebook, we have:
- Loaded FAIR^2 protein mass spectrometry dataset metadata and records via the Croissant schema.
- Inspected available record sets and fields, referencing all by `@id` for reproducibility.
- Demonstrated basic data extraction, filtering, normalization, and exploratory visualization steps.

For more advanced analysis, continue to explore relationships between fields, leverage documentation in the Croissant schema, and adjust selection criteria for your bioscience or bioinformatics questions.